# Microsoft TRELLIS: Multi-View Image-to-3D GLB Pipeline (L4 GPU)

透過 Microsoft TRELLIS 原生多視角演算法 (run_multi_image) 將正、背、側三視圖轉換為高擬真 PBR 3D GLB 模型。

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

def run_cmd(cmd, cwd=None, env=None):
    print(f"執行: {' '.join(cmd)}", flush=True)
    merged_env = os.environ.copy()
    if env:
        merged_env.update(env)
    res = subprocess.run(cmd, cwd=cwd, env=merged_env, capture_output=True, text=True)
    if res.returncode != 0:
        print(f"❌ 指令失敗: {' '.join(cmd)}", flush=True)
        print("--- STDERR ---", flush=True)
        print(res.stderr[-4000:], flush=True)
        print("--- STDOUT ---", flush=True)
        print(res.stdout[-4000:], flush=True)
        raise RuntimeError(f"執行失敗: {res.stderr[-500:]}")
    return res.stdout

SETUP_MARKER = Path("/content/.trellis_setup_done")
ENV_DIR = Path("/content/trellis_env")
PYTHON_BIN = ENV_DIR / "bin" / "python"
TRELLIS_DIR = Path("/content/TRELLIS")

# 自動搜尋可用的 CUDA Toolkit 路徑
cuda_candidates = [Path("/usr/local/cuda"), Path("/usr/local/cuda-12.2"), Path("/usr/local/cuda-12.4"), Path("/usr/local/cuda-12.1"), Path("/usr/local/cuda-12.0"), Path("/usr/local/cuda-11.8")]
cuda_home = "/usr/local/cuda"
for c in cuda_candidates:
    if (c / "bin" / "nvcc").exists():
        cuda_home = str(c)
        break

print(f"🔍 檢測到 CUDA Toolkit: {cuda_home}", flush=True)
build_env = os.environ.copy()
build_env["CUDA_HOME"] = cuda_home
build_env["PATH"] = f"{ENV_DIR}/bin:{cuda_home}/bin:" + build_env.get("PATH", "")
build_env["LD_LIBRARY_PATH"] = f"{cuda_home}/lib64:" + build_env.get("LD_LIBRARY_PATH", "")
build_env["TORCH_CUDA_ARCH_LIST"] = "8.9;8.0;7.5"
build_env["FORCE_CUDA"] = "1"

if not SETUP_MARKER.exists():
    print("🚀 正在配置 Python 3.10 與 Microsoft TRELLIS 核心環境 (L4 GPU)...", flush=True)
    
    # 0. 安裝 Headless OpenGL / PyVista 必要系統庫
    print("安裝系統 xvfb 與 libgl1...", flush=True)
    run_cmd(["apt-get", "update", "-qq"])
    run_cmd(["apt-get", "install", "-y", "-qq", "xvfb", "libgl1", "libglib2.0-0"])
    
    # 1. 下載 micromamba 輕量環境管理器
    mamba_bin = Path("/usr/local/bin/micromamba")
    if not mamba_bin.exists():
        print("下載 micromamba...", flush=True)
        run_cmd(["curl", "-Ls", "https://micro.mamba.pm/api/micromamba/linux-64/latest", "-o", "/tmp/mamba.tar.bz2"])
        run_cmd(["tar", "-xvjf", "/tmp/mamba.tar.bz2", "-C", "/usr/local/bin", "--strip-components=1", "bin/micromamba"])
    
    # 2. 建立 Python 3.10 隔離環境 (確保 spconv-cu120 官方 wheel 原生相容)
    if not ENV_DIR.exists():
        print("建立 Python 3.10 獨立環境...", flush=True)
        run_cmd([str(mamba_bin), "create", "-y", "-p", str(ENV_DIR), "-c", "conda-forge", "python=3.10", "pip", "ninja", "git"])
    
    # 3. 安裝 PyTorch 2.4.0 (CUDA 12.1 相容版) 與 xformers
    print("安裝 PyTorch 2.4.0 (cu121) 與 xformers...", flush=True)
    run_cmd([str(PYTHON_BIN), "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel", "ninja"], env=build_env)
    run_cmd([str(PYTHON_BIN), "-m", "pip", "install", "torch==2.4.0", "torchvision==0.19.0", "--index-url", "https://download.pytorch.org/whl/cu121"], env=build_env)
    run_cmd([str(PYTHON_BIN), "-m", "pip", "install", "xformers==0.0.27.post2", "--index-url", "https://download.pytorch.org/whl/cu121"], env=build_env)
    run_cmd([str(PYTHON_BIN), "-m", "pip", "install", "spconv-cu120"], env=build_env)
    run_cmd([str(PYTHON_BIN), "-m", "pip", "install", "kaolin", "-f", "https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.4.0_cu121.html"], env=build_env)
    
    # 3.1 補丁 PyTorch cpp_extension: 繞過 Colab 宿主機 nvcc (13.0) 與 PyTorch (12.1) 的版本檢驗阻擋
    cpp_ext_list = list(ENV_DIR.glob("lib/python3.*/site-packages/torch/utils/cpp_extension.py"))
    if cpp_ext_list:
        cpp_ext_path = cpp_ext_list[0]
        print(f"🔧 套用 PyTorch CUDA mismatch 補丁: {cpp_ext_path}", flush=True)
        txt = cpp_ext_path.read_text(encoding="utf-8")
        txt = txt.replace("raise RuntimeError(CUDA_MISMATCH_MESSAGE.format(cuda_str_version, torch.version.cuda))", "pass # CUDA_MISMATCH_BYPASS")
        cpp_ext_path.write_text(txt, encoding="utf-8")
    
    # 4. 安裝 TRELLIS 核心依賴套件 (鎖定 transformers<=4.46.3 以完全相容 PyTorch 2.4)
    print("安裝 TRELLIS 核心依賴套件...", flush=True)
    run_cmd([str(PYTHON_BIN), "-m", "pip", "install", "numpy<2.0", "pillow", "imageio", "imageio-ffmpeg", "tqdm", "easydict", "opencv-python-headless", "scipy", "rembg", "onnxruntime", "trimesh", "open3d", "xatlas", "pyvista", "pymeshfix", "igraph", "transformers<=4.46.3", "huggingface_hub", "pymeshlab", "timm", "plyfile"], env=build_env)
    run_cmd([str(PYTHON_BIN), "-m", "pip", "install", "git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8"], env=build_env)
    
    # 5. 編譯安裝 CUDA 擴展 (nvdiffrast, diffoctreerast, mip-splatting)
    print("安裝 CUDA 渲染擴展...", flush=True)
    nv_dir = Path("/tmp/extensions/nvdiffrast")
    if not nv_dir.exists():
        run_cmd(["git", "clone", "https://github.com/NVlabs/nvdiffrast.git", str(nv_dir)])
    run_cmd([str(PYTHON_BIN), "-m", "pip", "install", "--no-build-isolation", "-v", str(nv_dir)], env=build_env)
    
    diff_dir = Path("/tmp/extensions/diffoctreerast")
    if not diff_dir.exists():
        run_cmd(["git", "clone", "--recurse-submodules", "https://github.com/JeffreyXiang/diffoctreerast.git", str(diff_dir)])
    run_cmd([str(PYTHON_BIN), "-m", "pip", "install", "--no-build-isolation", "-v", str(diff_dir)], env=build_env)
    
    mip_dir = Path("/tmp/extensions/mip-splatting")
    if not mip_dir.exists():
        run_cmd(["git", "clone", "https://github.com/autonomousvision/mip-splatting.git", str(mip_dir)])
    run_cmd([str(PYTHON_BIN), "-m", "pip", "install", "--no-build-isolation", "-v", str(mip_dir / "submodules" / "diff-gaussian-rasterization")], env=build_env)
    
    # 6. 下載 TRELLIS 核心倉儲 (包含所有子模組如 flexicubes)
    if not (TRELLIS_DIR / "setup.sh").exists():
        if TRELLIS_DIR.exists():
            shutil.rmtree(str(TRELLIS_DIR), ignore_errors=True)
        run_cmd(["git", "clone", "--recurse-submodules", "https://github.com/microsoft/TRELLIS.git", str(TRELLIS_DIR)])
    else:
        run_cmd(["git", "submodule", "update", "--init", "--recursive"], cwd=str(TRELLIS_DIR))
    
    SETUP_MARKER.touch()
    print("✅ TRELLIS Python 3.10 核心環境配置就緒！", flush=True)
else:
    # 即使重用環境，確保 flexicubes 等子模組已遞歸初始化與 kaolin 安裝
    run_cmd(["git", "submodule", "update", "--init", "--recursive"], cwd=str(TRELLIS_DIR))
    run_cmd([str(PYTHON_BIN), "-m", "pip", "install", "transformers<=4.46.3"], env=build_env)
    run_cmd([str(PYTHON_BIN), "-m", "pip", "install", "kaolin", "-f", "https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.4.0_cu121.html"], env=build_env)
    print("✅ 重用既有 TRELLIS 軟體環境。", flush=True)


In [ ]:
import os, subprocess, sys
from pathlib import Path

ENV_DIR = Path("/content/trellis_env")
PYTHON_BIN = ENV_DIR / "bin" / "python"

# 撰寫隔離推論執行腳本
script_content = '''
import os, sys
os.environ["ATTN_BACKEND"] = "xformers"
os.environ["SPCONV_ALGO"] = "native"
os.environ["PYVISTA_OFF_SCREEN"] = "true"
sys.path.insert(0, "/content/TRELLIS")

from pathlib import Path
from PIL import Image

from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.utils import postprocessing_utils

front_path = os.environ.get("TRELLIS_FRONT_IMAGE", os.environ.get("HY3D_FRONT_IMAGE", "/content/input_front.png"))
back_path = os.environ.get("TRELLIS_BACK_IMAGE", os.environ.get("HY3D_BACK_IMAGE", "/content/input_back.png"))
side_path = os.environ.get("TRELLIS_SIDE_IMAGE", os.environ.get("HY3D_SIDE_IMAGE", "/content/input_side.png"))
output_path = Path(os.environ.get("TRELLIS_OUTPUT_PATH", os.environ.get("HY3D_OUTPUT_PATH", "/content/output_model.glb")))
simplify = float(os.environ.get("TRELLIS_SIMPLIFY", "0.95"))
texture_size = int(os.environ.get("TRELLIS_TEXTURE_SIZE", "2048"))

print("📥 正在載入 Microsoft TRELLIS 模型 (JeffreyXiang/TRELLIS-image-large)...", flush=True)
pipeline = TrellisImageTo3DPipeline.from_pretrained("JeffreyXiang/TRELLIS-image-large")
pipeline.cuda()
print("✅ TRELLIS 模型已成功載入 GPU (L4 VRAM 就緒)！", flush=True)

print(f"🖼️  讀取輸入視角圖片:\\n   - 正面: {front_path}\\n   - 背面: {back_path}\\n   - 側面: {side_path}", flush=True)
img_front = Image.open(front_path)
img_back = Image.open(back_path)
img_side = Image.open(side_path)

print("🚀 執行 TRELLIS 多視角神經潛空間擴散 (run_multi_image)...", flush=True)
images = [img_front, img_side, img_back]

outputs = pipeline.run_multi_image(
    images,
    seed=42,
    mode="stochastic",
    formats=["mesh", "gaussian"],
    sparse_structure_sampler_params={
        "steps": 12,
        "cfg_strength": 7.5,
    },
    slat_sampler_params={
        "steps": 12,
        "cfg_strength": 3.0,
    },
)

print("📦 正在自 3D Gaussian 與神經網格烘焙 PBR GLB 模型...", flush=True)
glb = postprocessing_utils.to_glb(
    app_rep=outputs["gaussian"][0],
    mesh=outputs["mesh"][0],
    simplify=simplify,
    texture_size=texture_size,
)

output_path.parent.mkdir(parents=True, exist_ok=True)
glb.export(str(output_path))
print(f"🎉 模型成功生成並導出至: {output_path} (大小: {output_path.stat().st_size / (1024*1024):.2f} MB)", flush=True)
'''

runner_script = Path("/content/run_trellis.py")
runner_script.write_text(script_content, encoding="utf-8")

print("🚀 在 Python 3.10 環境中啟動 TRELLIS 推論管線...", flush=True)
env = os.environ.copy()
env["PATH"] = f"{ENV_DIR}/bin:" + env.get("PATH", "")
p = subprocess.Popen([str(PYTHON_BIN), "-u", str(runner_script)], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end="", flush=True)
ret = p.wait()
if ret != 0:
    raise RuntimeError(f"TRELLIS 推論腳本執行失敗 (代碼 {ret})")
print("✅ TRELLIS 推論成功結束！", flush=True)
